# Parameters

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd

# ===== CONFIGURAÇÃO DE CAMINHOS =====
current_notebook = Path.cwd()  
project_root = current_notebook.parent.parent 

# Adiciona o diretório raiz ao sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Adiciona o diretório Modules ao sys.path
modules_dir = project_root / "Modules"
if str(modules_dir) not in sys.path:
    sys.path.insert(0, str(modules_dir))

# ===== IMPORTS DOS MÓDULOS =====
import Modules.ClusterDBSCANModule as cluster
import Modules.FutureAnalysisModule as fa
from Modules.SHAPClassifierModule import *
from Modules.config import CONFIG

# ===== CONFIGURAÇÕES DO PROJETO =====
DATAPATH = CONFIG["datapath"]
COVID_TRAIN_DATA_FILE = CONFIG["covid_train_data_file"]
COVID_TEST_DATA_FILE = CONFIG["covid_test_data_file"]
FUTURE_DATA_FILE = CONFIG["future_data_file"]

CONTROL_GROUP_TRAIN = CONFIG["control_group_train"]
CONTROL_GROUP_TEST = CONFIG["control_group_test"]
CONTROL_GROUP_READMISSION = CONFIG["control_group_readmission"]

FIGSIZE_CLUSTER_HEATMAP = CONFIG["figsize_cluster_heatmap"]
FIGSIZE_FUTURE_HEATMAP = CONFIG["figsize_future_heatmap"]
IMAGES_SAVE_PATH = CONFIG["image_save_path"]
TRIALS_OPTUNA = 250

# Import data

In [ ]:
# ===== CARREGAMENTO DOS DADOS =====
data_folder = current_notebook / DATAPATH

covid_train = pd.read_csv(data_folder / COVID_TRAIN_DATA_FILE)
covid_test = pd.read_csv(data_folder / COVID_TEST_DATA_FILE)
future_data = pd.read_csv(data_folder / FUTURE_DATA_FILE)

control_train = pd.read_csv(data_folder / CONTROL_GROUP_TRAIN)
control_test = pd.read_csv(data_folder / CONTROL_GROUP_TEST)
control = pd.concat([control_train, control_test], axis=0)
control_readmission = pd.read_csv(data_folder / CONTROL_GROUP_READMISSION)

# Feature engineering: morte após a internação
covid_train['died_after'] = ((covid_train['died'] == 1) & (covid_train['died_in_stay'] == 0)).astype(int)
covid_test['died_after'] = ((covid_test['died'] == 1) & (covid_test['died_in_stay'] == 0)).astype(int)
future_data['died_after'] = ((future_data['died'] == 1) & (future_data['died_in_stay'] == 0)).astype(int)

shap.initjs()

## Get Top features

In [ ]:
features2remove = [
    "died_in_stay",
    "COVID",
    "charlson_comorbidity_index"
]  # Add the features to remove

In [ ]:
helper = ShapHelperClassifier(
    covid_train.drop(columns=["subject_id", "hadm_id"]),
    covid_test.drop(columns=["subject_id", "hadm_id"]),
    "died",
)
helper.remove_features(features2remove)

In [ ]:
param = {
    "learning_rate": 0.03194788423314631,
    "n_estimators": 855,
    "gamma": 0.15986032691615765,
    "reg_alpha": 0.7169178389895312,
    "reg_lambda": 0.5826947399857607,
    "scale_pos_weight": 12,
}

In [ ]:
helper.train_single_model(param, model_name="xgboost")
helper.print_metrics()
helper.show_confusion_matrix()

In [ ]:
helper.compute_shap_values()

In [ ]:
features = helper.get_top_features(20)

In [ ]:
helper.plot_shap_summary()

In [ ]:
features

## Setup Hierarchical Clustering

In [ ]:
data_covid = pd.concat([covid_train, covid_test], axis=0)
data_covid = data_covid.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
features_not_considered = [x for x in data_covid.columns.tolist() if x not in features]

In [ ]:
helper = cluster.DBSCANClusterHelper(data=data_covid, features_not_considered=features_not_considered)

## Find best hyperparameters for DBSCAN

In [ ]:
param = {
    "eps": {"min": 1e-3, "max": 10},
    "min_samples": {"min": 1, "max": 100}
}

### DBCV

In [ ]:
os.environ["PYTHONWARNINGS"] = "ignore"
dbcv_df, dbcv_param, dbcv_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="dbcv",
    suffix="death"
)

In [ ]:
dbcv_param = {'eps': 2.6603265710919337, 'min_samples': 49}

### DISCO

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
disco_df, disco_param, disco_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="disco",
    suffix="death"
)

In [ ]:
disco_param = {'eps': 2.56219076513391, 'min_samples': 23}

### DSI

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
dsi_df, dsi_param, dsi_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="dsi",
    suffix="death"
)

In [ ]:
dsi_param = {'eps': 2.6071589806556856, 'min_samples': 37}

### Silhouette

In [ ]:
os.environ['PYTHONWARNINGS'] = 'ignore'
silhouette_df, silhouette_param, silhouette_best = helper.optuna_grid_search(
    parameters=param, 
    n_trials=TRIALS_OPTUNA, 
    save_storage=True, 
    metric="silhouette",
    suffix="death"
)

In [ ]:
silhouette_param = {'eps': 2.7078464768336747, 'min_samples': 67}

### Metrics

In [ ]:
helper.clustering(eps=dbcv_param["eps"], min_samples=dbcv_param["min_samples"])
helper.get_metrics()

In [ ]:
helper.clustering(eps=disco_param["eps"], min_samples=disco_param["min_samples"])
helper.get_metrics()

In [ ]:
helper.clustering(eps=dsi_param["eps"], min_samples=dsi_param["min_samples"])
helper.get_metrics()

In [ ]:
helper.clustering(eps=silhouette_param["eps"], min_samples=silhouette_param["min_samples"])
helper.get_metrics()

## Best Result - DISCO

In [ ]:
helper.clustering(eps=disco_param['eps'], min_samples=disco_param['min_samples'])
helper.get_metrics()

In [ ]:
helper.heatmap_clusters_categorical(
    figsize=FIGSIZE_CLUSTER_HEATMAP,
    savepath=IMAGES_SAVE_PATH + "dbscan-death-categorical"
)

In [ ]:
selected_clusters = [-1,0]

In [ ]:
helper.show_cluster_compare_numerical(
    selected_clusters=selected_clusters,
    figsize=(10, 15),
    savepath=IMAGES_SAVE_PATH + "dbscan-death-numerical")

In [ ]:
helper.set_clustered_autoencoder()

In [ ]:
helper.show_clustered_autoencoder(selected_clusters=selected_clusters, savepath=IMAGES_SAVE_PATH + "dbscan-autoencoder-death")

##### Future data


In [ ]:
future_helper = fa.FutureAnalysisHelper(
    helper.clusteredData, future_data, control, control_readmission
)
delta = future_helper.get_delta_clusters(percentage=True, relative_total=True)
future_helper.show_delta_heatmap(
    figsize=FIGSIZE_FUTURE_HEATMAP,
    relative_total=True,
    selected_clusters=selected_clusters,
    # savepath=IMAGES_SAVE_PATH + "dbscan-death-future",
)

In [ ]:
future_helper.get_mean_readmission()

In [ ]:
future_helper.get_mean_days_gap()

In [ ]:
future_helper.get_mortality_rates(only_first_admission=True)

# Add Log

In [ ]:
log_file = "../metrics.csv"
current_dir = os.getcwd()
log_file_path = os.path.join(current_dir, log_file)

metrics = helper.get_metrics()

# Add line to save log
if os.path.exists(log_file_path):
    with open(log_file_path, 'a') as f:
        f.write(f"DBSCAN, Shap, Comprehensive, {metrics['disco']}, {metrics['dbcv']}, {metrics['dsi']}, {metrics['silhouette']}\n")